# Role-Aware SAAMR: SDF to OpenFF/OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook starts from SDF files generated by `Role_Aware_SAAMR_Quickstart.ipynb` and demonstrates the downstream handoff:

```text
Primitive -> RDKit -> SDF -> OpenFF -> OpenMM
```

Generated simulation artifacts are written under `examples_system/role_aware_saamr_outputs/` and are ignored by git.

## 1. Environment and Inputs

In [ ]:
from pathlib import Path
import shutil
import sys

import numpy as np


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

SDF_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "sdf"
SIM_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "openmm"
SIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"SDF directory: {SDF_DIR}")
print(f"Simulation output directory: {SIM_DIR}")

## 2. Load SDF Files with RDKit

In [ ]:
from rdkit import Chem

sdf_paths = sorted(SDF_DIR.glob("psu_pes_chain_*.sdf"))
if not sdf_paths:
    raise FileNotFoundError(
        f"No SDF files found in {SDF_DIR}. Run Role_Aware_SAAMR_Quickstart.ipynb first."
    )

rdkit_mols = []
for path in sdf_paths:
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    mol = supplier[0]
    if mol is None:
        raise ValueError(f"Could not read {path}")
    Chem.SanitizeMol(Chem.Mol(mol))
    rdkit_mols.append(mol)

print(f"Loaded {len(rdkit_mols)} SDF molecule(s)")
for path, mol in zip(sdf_paths, rdkit_mols):
    print(f"  {path.name}: atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}")

## 3. Reconstruct MuPT SAAMR Hierarchies

This confirms that SDF files still contain enough information to recover the role-aware MuPT hierarchy.

In [ ]:
from mupt.interfaces.rdkit import primitive_from_mupt_sdf
from mupt.roles import PrimitiveRole

reconstructed = primitive_from_mupt_sdf(
    sdf_paths,
    reconstruct_bonds=False,
    reconstruct_shapes=False,
)

assert reconstructed.role == PrimitiveRole.UNIVERSE
assert len(reconstructed.children) == len(sdf_paths)
for idx, segment in enumerate(reconstructed.children):
    assert segment.role == PrimitiveRole.SEGMENT
    assert all(residue.role == PrimitiveRole.RESIDUE for residue in segment.children)
    print(
        f"reconstructed segment {idx}: residues={len(segment.children)}, "
        f"particles={len(segment.leaves)}"
    )

## 4. OpenFF/OpenMM Availability and Run Settings

The workflow below mirrors the OpenMM section of the random copolymer quickstart: each molecule is charged with an OpenFF graph neural network model, packed into a target-density box with OpenFF Packmol, parameterized as one `Interchange`, minimized, serialized, and optionally run for a short trajectory.

The charge model is `openff-gnn-am1bcc-1.0.0.pt`, avoiding per-molecule AM1-BCC calculations. This model is provided by OpenFF NAGL, not RDKit, AmberTools, or the built-in OpenFF toolkit wrappers. If NAGL is unavailable, the parameterization cell skips with installation instructions rather than falling back to AM1-BCC.

The random-walk coordinates stored in the SDF files are only used to build polymer conformations. They are not melt packing coordinates. Before simulation, this notebook repacks the OpenFF molecules with Packmol so PME sees a physically sized periodic box instead of a huge sparse bounding box.

In [ ]:
try:
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        # Importing the wrapper is not enough: NAGLToolkitWrapper can be present
        # in OpenFF Toolkit even when the openff-nagl package is not installed.
        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError(
                "OpenFF Toolkit provides NAGLToolkitWrapper, but the OpenFF NAGL "
                "backend is unavailable. Install openff-nagl. See "
                "https://docs.openforcefield.org/projects/nagl/en/latest/installation.html"
            )
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

PACKMOL_AVAILABLE = False
PACKMOL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.packmol import PACKMOLRuntimeError, pack_box

        PACKMOL_AVAILABLE = shutil.which("packmol") is not None
        if not PACKMOL_AVAILABLE:
            PACKMOL_IMPORT_ERROR = RuntimeError(
                "openff-packmol is importable, but the packmol executable "
                "is not on PATH. Install packmol from conda-forge."
            )
    except ModuleNotFoundError as exc:
        PACKMOL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"

RUN_OPENFF_PARAMETERIZATION = OPENFF_AVAILABLE and NAGL_AVAILABLE and PACKMOL_AVAILABLE
RUN_OPENMM_MINIMIZATION = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_OPENMM_DYNAMICS = False  # Set True for a short trajectory after minimization.
ENSEMBLE = "NVT"  # Choose "NVT" or "NPT". NPT is for larger packed systems.
PACKMOL_ATTEMPTS = [
    {"target_density_g_per_ml": 0.60, "tolerance_nm": 0.10},
    {"target_density_g_per_ml": 0.50, "tolerance_nm": 0.10},
    {"target_density_g_per_ml": 0.40, "tolerance_nm": 0.08},
    {"target_density_g_per_ml": 0.30, "tolerance_nm": 0.08},
]

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenFF Packmol available: {PACKMOL_AVAILABLE}")
if not PACKMOL_AVAILABLE and PACKMOL_IMPORT_ERROR is not None:
    print(f"  {PACKMOL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Partial charge model: {PARTIAL_CHARGE_METHOD}")
print(f"Packmol attempts: {PACKMOL_ATTEMPTS}")
if ENSEMBLE not in {"NVT", "NPT"}:
    raise ValueError("ENSEMBLE must be either 'NVT' or 'NPT'")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")
print(f"Dynamics ensemble: {ENSEMBLE}")

## 5. Convert RDKit Molecules to OpenFF Molecules

In [ ]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol: Molecule) -> None:
    """Copy SDF atom-property metadata into OpenFF atom metadata."""
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        residue_name = str(props.get("residue_name", props.get("mupt_residue_label", "UNK")))
        residue_number = str(props.get("residue_id", props.get("mupt_residue_index", "1")))
        chain_id = str(props.get("chain_id", "A"))

        off_atom.metadata.update({
            "residue_name": residue_name,
            "residue_number": residue_number,
            "chain_id": chain_id,
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


off_molecules = []
if OPENFF_AVAILABLE:
    for mol in rdkit_mols:
        off_mol = Molecule.from_rdkit(
            mol,
            allow_undefined_stereo=True,
            hydrogens_are_explicit=True,
        )
        transfer_rdkit_metadata_to_openff(mol, off_mol)
        off_molecules.append(off_mol)
    print(f"Created {len(off_molecules)} OpenFF Molecule object(s)")
    first_atom = off_molecules[0].atom(0)
    print("First OpenFF atom metadata:")
    print(f"  residue_name: {first_atom.metadata.get('residue_name')}")
    print(f"  residue_number: {first_atom.metadata.get('residue_number')}")
    print(f"  chain_id: {first_atom.metadata.get('chain_id')}")
else:
    print("Skipping OpenFF conversion because openff-toolkit is not installed.")


## 6. Packmol Packing and GNN-Charged OpenFF Parameterization

This cell assigns charges per molecule with the OpenFF GNN model, tries a small schedule of Packmol packing settings, then parameterizes the best packed topology as one `Interchange`. This avoids running slow AM1-BCC calculations for every chain and avoids the sparse bounding-box PME grid that can trigger CUDA out-of-memory errors. Packmol can fail for long polymers; when it leaves a candidate `packmol_output.pdb`, the notebook uses those best coordinates as a pragmatic starting point for energy minimization.

The important detail is the explicit `NAGLToolkitWrapper` registry. Without it, OpenFF only tries RDKit, AmberTools, and the built-in toolkit wrappers, none of which provide `openff-gnn-am1bcc-1.0.0.pt`. If NAGL is unavailable, the cell skips rather than using AM1-BCC.

In [ ]:
def load_packmol_output_positions(pdb_path: Path):
    """Load atom coordinates from Packmol PDB output in angstrom."""
    positions = []
    for line in pdb_path.read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            positions.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    if not positions:
        raise ValueError(f"No atom positions found in {pdb_path}")
    return np.asarray(positions, dtype=float) * off_unit.angstrom


def rectangular_box_from_density(molecules, target_density_g_per_ml: float):
    """Return cubic box vectors from total molecular mass and target density."""
    target_density = target_density_g_per_ml * off_unit.gram / off_unit.milliliter
    total_mass = sum(sum(atom.mass for atom in molecule.atoms) for molecule in molecules)
    volume = (total_mass / target_density).m_as(off_unit.nanometer**3)
    length = volume ** (1.0 / 3.0)
    return np.diag([length, length, length]) * off_unit.nanometer


def topology_from_packmol_candidate(molecules, pdb_path: Path, target_density_g_per_ml: float):
    """Build an OpenFF topology from Packmol's best written coordinates."""
    topology = Topology.from_molecules(molecules)
    positions = load_packmol_output_positions(pdb_path)
    if positions.shape[0] != topology.n_atoms:
        raise ValueError(
            f"Packmol candidate atom count mismatch: {positions.shape[0]} positions "
            f"for {topology.n_atoms} topology atoms"
        )
    topology.set_positions(positions)
    topology.box_vectors = rectangular_box_from_density(molecules, target_density_g_per_ml)
    return topology


interchange = None
packed_topology = None
candidate_topology = None
packmol_status = None

if OPENFF_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for mol_idx, off_mol in enumerate(off_molecules):
        print(f"Charging molecule {mol_idx + 1}/{len(off_molecules)}")
        off_mol.assign_partial_charges(
            partial_charge_method=PARTIAL_CHARGE_METHOD,
            toolkit_registry=nagl_registry,
        )

    packmol_root = SIM_DIR / "packmol_working_files"
    packmol_root.mkdir(parents=True, exist_ok=True)
    for attempt_idx, attempt in enumerate(PACKMOL_ATTEMPTS, start=1):
        target_density = attempt["target_density_g_per_ml"] * off_unit.gram / off_unit.milliliter
        tolerance = attempt["tolerance_nm"] * off_unit.nanometer
        packmol_dir = packmol_root / f"attempt_{attempt_idx:02d}"
        print(
            f"Packmol attempt {attempt_idx}/{len(PACKMOL_ATTEMPTS)}: "
            f"density={attempt['target_density_g_per_ml']} g/mL, "
            f"tolerance={attempt['tolerance_nm']} nm"
        )
        try:
            packed_topology = pack_box(
                molecules=off_molecules,
                number_of_copies=[1] * len(off_molecules),
                target_density=target_density,
                tolerance=tolerance,
                working_directory=str(packmol_dir),
                retain_working_files=True,
            )
            packmol_status = f"success on attempt {attempt_idx}"
            break
        except PACKMOLRuntimeError as exc:
            candidate_path = packmol_dir / "packmol_output.pdb"
            if candidate_path.exists() and candidate_path.stat().st_size > 0:
                candidate_topology = topology_from_packmol_candidate(
                    off_molecules,
                    candidate_path,
                    attempt["target_density_g_per_ml"],
                )
                packmol_status = f"candidate retained from failed attempt {attempt_idx}: {exc}"
                print("Retained Packmol candidate output; continuing attempts for a clean success.")
                continue
            print(f"Packmol attempt {attempt_idx} failed without candidate output: {exc}")

    if packed_topology is None and candidate_topology is not None:
        packed_topology = candidate_topology
        print("Using retained Packmol candidate output for minimization despite nonzero exit.")

    if packed_topology is None:
        raise RuntimeError(
            "All Packmol attempts failed without producing candidate coordinates. "
            f"Inspect retained files under {packmol_root}."
        )

    interchange = ff.create_interchange(
        packed_topology,
        charge_from_molecules=off_molecules,
    )
    print(f"Packed interchange with {interchange.topology.n_atoms} atoms")
    print(f"Packmol status: {packmol_status}")
    print("Packed box vectors (nm):")
    print(interchange.box.m_as(off_unit.nanometer))
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print(
        "OpenFF parameterization skipped because OpenFF NAGL is unavailable. "
        "Install openff-nagl to use openff-gnn-am1bcc-1.0.0.pt. "
        "This tutorial intentionally does not fall back to AM1-BCC because "
        "AM1-BCC is slow for polymer chains."
    )
elif OPENFF_AVAILABLE and not PACKMOL_AVAILABLE:
    print(
        "OpenFF parameterization skipped because OpenFF Packmol is unavailable. "
        "Install openff-packmol and the packmol executable from conda-forge."
    )
else:
    print("OpenFF parameterization skipped. Install openff-toolkit, openff-nagl, openff-packmol, and packmol to run it.")

## 7. Packed Coordinate Diagnostics

Packmol supplies the simulation coordinates and periodic box. This cell performs lightweight diagnostics on those packed coordinates without constructing an O(N²) distance matrix.

In [ ]:
def minimum_pair_distance_nm(positions_nm: np.ndarray) -> float:
    """Return the nearest-neighbor distance without an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions_nm).query(positions_nm, k=2)
    return float(np.min(distances[:, 1]))


if interchange is not None:
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    min_distance_nm = minimum_pair_distance_nm(positions_nm)
    box_lengths = np.linalg.norm(interchange.box.m_as(off_unit.nanometer), axis=1)
    box_volume_per_atom = float(abs(np.linalg.det(interchange.box.m_as(off_unit.nanometer))) / positions_nm.shape[0])
    print(f"Minimum packed atom-atom distance: {min_distance_nm:.4f} nm")
    print(
        f"Packed box vector lengths: {box_lengths[0]:.2f} x "
        f"{box_lengths[1]:.2f} x {box_lengths[2]:.2f} nm"
    )
    print(f"Packed box volume per atom: {box_volume_per_atom:.3f} nm^3/atom")
    if min_distance_nm < 0.005:
        raise ValueError(
            "Detected overlapping or nearly overlapping packed coordinates before OpenMM. "
            "Increase PACKMOL_TOLERANCE_NM or inspect Packmol output."
        )
else:
    print("Packed coordinate diagnostics skipped because no Interchange was created.")


## 8. OpenMM Minimization and Optional Short Dynamics

This cell creates OpenMM simulation components from the Packmol-packed periodic box, runs minimization, serializes the result, and optionally runs a short trajectory. `ENSEMBLE = "NVT"` uses a Langevin integrator without a barostat. `ENSEMBLE = "NPT"` adds a Monte Carlo barostat and should only be used after minimization/equilibration checks, because the box must remain larger than twice the nonbonded cutoff in every dimension.

In [ ]:
simulation = None
state = None
openmm_dir = SIM_DIR / "OpenMM"
openmm_dir.mkdir(parents=True, exist_ok=True)

if interchange is not None and OPENMM_AVAILABLE and RUN_OPENMM_MINIMIZATION:
    temperature = 300.0 * omm_unit.kelvin
    pressure = 1.0 * omm_unit.atmosphere
    time_step = 2.0 * omm_unit.femtosecond
    friction = 1.0 / omm_unit.picosecond
    n_steps = 250
    report_interval = 25

    integrator = LangevinMiddleIntegrator(temperature, friction, time_step)
    additional_forces = []
    if ENSEMBLE == "NPT":
        # The barostat changes box dimensions. OpenMM requires every periodic box
        # length to remain larger than twice the nonbonded cutoff; use NPT only
        # after the packed melt has minimized and equilibrated sensibly.
        additional_forces.append(MonteCarloBarostat(pressure, temperature, 25))
        print("Using NPT dynamics with a Monte Carlo barostat")
    else:
        print("Using NVT dynamics without a barostat")

    simulation = interchange.to_openmm_simulation(
        integrator=integrator,
        combine_nonbonded_forces=False,
        additional_forces=additional_forces,
    )

    print("Running energy minimization...")
    simulation.minimizeEnergy()
    state = simulation.context.getState(getEnergy=True, getPositions=True)
    print(f"Minimized potential energy: {state.getPotentialEnergy()}")

    system_name = "role_aware_saamr"
    topology_path = openmm_dir / f"{system_name}_topology.pdb"
    system_path = openmm_dir / f"{system_name}_system.xml"
    state_path = openmm_dir / f"{system_name}_state.xml"
    integrator_path = openmm_dir / f"{system_name}_integrator.xml"

    with topology_path.open("w") as handle:
        PDBFile.writeFile(simulation.topology, state.getPositions(asNumpy=True), handle)
    system_path.write_text(XmlSerializer.serialize(simulation.system))
    state_path.write_text(XmlSerializer.serialize(state))
    integrator_path.write_text(XmlSerializer.serialize(integrator))

    print("Serialized OpenMM components:")
    for path in (topology_path, system_path, state_path, integrator_path):
        print(f"  {path.relative_to(EXAMPLES_ROOT)}")

    if RUN_OPENMM_DYNAMICS:
        dcd_path = openmm_dir / f"{system_name}_{ENSEMBLE.lower()}_trajectory.dcd"
        state_data_path = openmm_dir / f"{system_name}_{ENSEMBLE.lower()}_state_data.csv"
        simulation.reporters.append(DCDReporter(str(dcd_path), report_interval))
        simulation.reporters.append(
            StateDataReporter(
                str(state_data_path),
                reportInterval=report_interval,
                step=True,
                time=True,
                potentialEnergy=True,
                kineticEnergy=True,
                temperature=True,
                volume=True,
                density=True,
                speed=True,
            )
        )
        print(f"Running {n_steps} steps of {ENSEMBLE} dynamics...")
        simulation.step(n_steps)
        print(f"Trajectory saved to: {dcd_path.relative_to(EXAMPLES_ROOT)}")
else:
    print("OpenMM minimization skipped. Create an Interchange and enable RUN_OPENMM_MINIMIZATION to run it.")

## 9. Optional Export Templates

These cells are templates for downstream exporters. They are disabled by default so tutorial execution does not create heavy MD artifacts.

In [ ]:
RUN_GROMACS_EXPORT = False
RUN_LAMMPS_EXPORT = False

if interchange is not None and RUN_GROMACS_EXPORT:
    gromacs_prefix = SIM_DIR / "role_aware_saamr"
    interchange.to_gromacs(str(gromacs_prefix), decimal=5)
    print(f"Wrote GROMACS files with prefix {gromacs_prefix}")
else:
    print("GROMACS export skipped.")

if interchange is not None and RUN_LAMMPS_EXPORT:
    print("LAMMPS export hook goes here once a project-standard exporter is selected.")
else:
    print("LAMMPS export skipped.")

## 10. Summary

This notebook is the downstream half of the workflow. The first notebook builds role-aware MuPT SAAMR systems and writes SDF files; this notebook loads those files, assigns GNN partial charges, parameterizes with OpenFF, and provides the OpenMM execution path used for simulation setup.